# 05 · Approximate nearest neighbors: szybciej za cenę ryzyka

Dokładny skan z lekcji 04 porównuje pytanie z każdym wektorem. Przy milionach fragmentów bywa drogi. ANN ogranicza liczbę porównań, lecz może pominąć prawdziwego sąsiada. Poznamy dwa mechanizmy: **inverted file (IVF)** przez prostą partycję k-means oraz **HNSW** w lokalnej, pamięciowej Chroma. PQ kompresuje wektory i zmniejsza pamięć kosztem błędu kwantyzacji; nie będziemy udawać, że nasz mały przykład mierzy produkcyjny zysk.

In [ ]:
# Jedyna komórka z importami i ścieżką projektu.
import sys
import time
from pathlib import Path

import chromadb
import numpy as np
from sklearn.cluster import KMeans

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, section_records

## 1. Złoty punkt odniesienia

Normalizujemy embeddingi i zapisujemy exact top-5 dla kilku pytań. `recall@5` ANN oznacza tu udział **pięciu dokładnych sąsiadów** znalezionych przez przybliżenie. Nie jest to jeszcze miara odpowiedzi na pytanie.

In [ ]:
records = section_records()
raw = embed_texts([row["text"] for row in records])
unit = raw / np.linalg.norm(raw, axis=1, keepdims=True)
questions = ["zero denominator in percentage forecast error", "November 2011 header billing",
             "credit note and receivables balance", "stockout effect on forecast backtest",
             "quality incident investigation evidence"]
q_raw = embed_texts(questions)
queries = q_raw / np.linalg.norm(q_raw, axis=1, keepdims=True)
exact = [np.argsort(-(unit @ q))[:5] for q in queries]
len(records), [records[int(i)]["id"] for i in exact[0]]

## 2. IVF w kilkunastu liniach

KMeans wyznacza centroidy. Po normalizacji przypisujemy dokumenty i wybieramy komórki pytania według tego samego cosinusa co ranking dokumentów. Pytanie najpierw wybiera `nprobe` centroidów, a potem porównuje się dokładnie tylko z ich członkami. Mniejszy `nprobe` oznacza mniej kandydatów i możliwy spadek recall. To dydaktyczny odpowiednik idei IVF, nie pełna implementacja FAISS.

In [ ]:
centroids = KMeans(n_clusters=8, random_state=42, n_init=10).fit(unit)
centroid_unit = centroids.cluster_centers_ / np.linalg.norm(centroids.cluster_centers_, axis=1, keepdims=True)
assignments = np.argmax(unit @ centroid_unit.T, axis=1)
ivf_results = []
for nprobe in (1, 2, 4, 8):
    recalls = []
    examined = []
    for q, gold in zip(queries, exact):
        centroid_scores = centroid_unit @ q
        chosen = np.argsort(-centroid_scores)[:nprobe]
        candidates = np.flatnonzero(np.isin(assignments, chosen))
        ranked = candidates[np.argsort(-(unit[candidates] @ q))[:5]]
        recalls.append(len(set(ranked) & set(gold)) / len(gold))
        examined.append(len(candidates))
    ivf_results.append({"nprobe": nprobe, "średnio_porównań": round(float(np.mean(examined)), 1),
                        "recall@5_vs_exact": round(float(np.mean(recalls)), 3)})
ivf_results

## 3. Chroma HNSW w pamięci

HNSW przechowuje graf sąsiedztwa i podczas wyszukiwania eksploruje jego część. `max_neighbors` wpływa na rozmiar grafu, `ef_search` na głębokość szukania. Chroma zwraca **cosine distance** (mniej = bliżej), podczas gdy nasz exact skan zwraca cosine similarity (więcej = bliżej). Na małym korpusie opóźnienie Chroma może być większe od pełnego skanu; nie wyciągaj wniosku o skalowaniu z tych milisekund.

In [ ]:
client = chromadb.EphemeralClient()
hnsw_results = []
for ef in (10, 100):
    collection = client.create_collection(f"rag-hnsw-ef-{ef}", configuration={"hnsw": {"space": "cosine", "ef_search": ef, "max_neighbors": 16}})
    collection.add(ids=[row["id"] for row in records], embeddings=unit.tolist(),
                   documents=[row["text"] for row in records])
    start = time.perf_counter()
    found = collection.query(query_embeddings=queries.tolist(), n_results=5, include=["distances"])
    elapsed = time.perf_counter() - start
    recalls = [len(set(ids) & {records[int(i)]["id"] for i in gold}) / 5 for ids, gold in zip(found["ids"], exact)]
    hnsw_results.append({"ef_search": ef, "recall@5_vs_exact": round(float(np.mean(recalls)), 3),
                         "czas_5_zapytań_ms": round(1000 * elapsed, 2), "pierwsza_odległość": round(found["distances"][0][0], 3)})
hnsw_results

## Kiedy używać której metody?

| Metoda | Koszt pamięci | Strojenie | Typowe ryzyko |
|---|---|---|---|
| Exact | pełne wektory | brak progu ANN | czas pełnego skanu |
| IVF | wektory + centroidy | `nlist`, `nprobe` | dobry sąsiad w nieodwiedzonej komórce |
| HNSW | wektory + graf | `M`, `efConstruction`, `efSearch` | zbyt płytka eksploracja |
| PQ | skompresowane kody | liczba części i bitów | błąd odległości po kwantyzacji |

**Ćwiczenie:** zmień `nprobe` i sprawdź pytanie, dla którego recall spadł. W praktyce oceniaj jednocześnie recall względem exact, trafność merytoryczną, opóźnienie, pamięć, aktualizacje i filtry dostępu. [FAISS index guide](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes) · [Chroma HNSW](https://cookbook.chromadb.dev/core/configuration/).